# Notebook 05: Distribution Grid Power Flow Simulation & Grid Impact Validation

**Objective:** Validate optimal EVCS placement against DPDC 33kV and DESCO 11kV distribution grid networks using pandapower AC power flow (Newton-Raphson), analyzing bus voltage stability ($0.95 \le V \le 1.05\text{ p.u.}$) and transformer/feeder thermal loading limits.

In [ ]:
import os
import sys
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.grid.power_flow import GridPowerFlowSimulator, plot_voltage_profile_comparison
from src.grid.voltage_stability import evaluate_voltage_stability, evaluate_thermal_loading
from src.visualization.pareto_front import find_knee_point_solution

dpdc_net_path = os.path.join(module_path, 'data', 'grid_models', 'dpdc_33kv_subnetwork.json')
sim = GridPowerFlowSimulator(dpdc_net_path)

# 1. Baseline Grid Simulation
base_bus, base_line, base_v_met, base_t_met = sim.run_baseline_power_flow()
print('Baseline Grid Results (No EV Load):')
print(f"  - Voltage Range: {base_v_met['min_v_pu']:.4f} to {base_v_met['max_v_pu']:.4f} p.u.")
print(f"  - Voltage Deviation Index: {base_v_met['vdi']:.5f}")
print(f"  - Max Feeder Line Loading: {base_t_met['max_line_loading_pct']:.1f}%")

## 1. Injecting Optimal EVCS Stations & Simulating AC Power Flow

In [ ]:
pareto_csv = os.path.join(module_path, 'results', 'tables', 'optimal_solutions_pareto.csv')
pareto_df = pd.read_csv(pareto_csv)
knee_idx, chosen_sol = find_knee_point_solution(pareto_df)

cand_path = os.path.join(module_path, 'data', 'processed', 'candidate_sites_filtered.geojson')
cand_gdf = gpd.read_file(cand_path)

selected_ids = chosen_sol['selected_station_ids'].split(';')
active_stations = cand_gdf[cand_gdf['candidate_id'].isin(selected_ids)].copy()
active_stations['total_grid_power_kw'] = float(chosen_sol['total_grid_power_kw']) / len(selected_ids)

sim.inject_ev_station_loads(active_stations)
ev_bus, ev_line, ev_v_met, ev_t_met = sim.run_ev_power_flow()

print(f"EV-Integrated Grid Results ({len(selected_ids)} Active Stations | Total Load: {chosen_sol['total_grid_power_kw']:.1f} kW):")
print(f"  - Voltage Range: {ev_v_met['min_v_pu']:.4f} to {ev_v_met['max_v_pu']:.4f} p.u.")
print(f"  - Voltage Deviation Index: {ev_v_met['vdi']:.5f}")
print(f"  - Max Feeder Line Loading: {ev_t_met['max_line_loading_pct']:.1f}%")
print(f"  - Compliant with Grid Limits: {ev_v_met['is_compliant'] and ev_t_met['is_thermal_compliant']}")

## 2. Voltage Profile Comparison Visualization

In [ ]:
fig_path = os.path.join(module_path, 'results', 'figures', 'voltage_profile_comparison.png')
plot_voltage_profile_comparison(base_bus, ev_bus, fig_path)

img = plt.imread(fig_path)
plt.figure(figsize=(11, 7))
plt.imshow(img)
plt.axis('off')
plt.show()